# Development-only additional-model comparison

This notebook reads the aggregate comparison of the five original fixed candidates, CatBoost and main-effects-only EBM. It does not train models, show participant records, access the previously evaluated test cohort, freeze artifacts or change deployment.

In [1]:
from pathlib import Path
import hashlib, json
import pandas as pd
from IPython.display import Markdown, display

def repository_root():
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / 'pyproject.toml').is_file() and (candidate / 'training').is_dir(): return candidate
    raise RuntimeError('Repository root not found')
report_path = repository_root() / 'reports' / 'generated' / 'development-additional-model-comparison-report.json'
report_bytes = report_path.read_bytes(); report = json.loads(report_bytes.decode('utf-8'))
assert report['scope'] == 'development_only_nested_additional_model_comparison'
assert report['final_test_evaluated'] is False
assert report['privacy']['participant_identifiers_written'] is False
assert report['privacy']['row_level_predictions_written'] is False
pd.DataFrame([{'Report': str(report_path.relative_to(repository_root())), 'SHA-256': hashlib.sha256(report_bytes).hexdigest().upper(), 'Development participants': report['development_participants'], 'Locked test evaluated': report['final_test_evaluated']}])

,Report,SHA-256,Development participants,Locked test evaluated
0,reports\generated\development-additional-model...,7F726DFFA17DA318949FB5ACD34B8EB773F75A8D019497...,500,False


## Methodology and configurations

In [2]:
pd.DataFrame([{**report['cross_validation'], 'Primary metric': report['primary_selection_metric'], 'Calibration': report['calibration'], 'Threshold selection': report['threshold_selection']}])

,outer_splits,outer_repeats,inner_splits,shared_fold_plan_sha256,same_outer_folds_for_all_domains_and_candidates,Primary metric,Calibration,Threshold selection
0,5,5,3,7AB8604BB0DD59DA33420D2743730A468D22C06FFD7EE1...,True,average_precision_pr_auc,inner_oof_regularized_logistic_sigmoid_on_raw_...,inner_oof_maximize_balanced_accuracy


## Domain rankings by repeated-validation PR-AUC

In [3]:
domain_labels = {'target_communication_concern': 'Communication', 'target_cognitive_concern': 'Cognitive', 'target_social_emotional_concern': 'Social-emotional'}
ranking_rows = []
for key, domain in report['domains'].items():
    for rank, row in enumerate(domain['ranking'], start=1): ranking_rows.append({'Domain': domain_labels[key], 'Rank': rank, **row})
pd.DataFrame(ranking_rows).style.format({'mean_average_precision': '{:.4f}', 'average_precision_standard_deviation': '{:.4f}', 'mean_roc_auc': '{:.4f}', 'mean_brier_score': '{:.4f}'}).hide(axis='index')

Domain,Rank,candidate,mean_average_precision,average_precision_standard_deviation,mean_roc_auc,mean_brier_score
Communication,1,xgboost_regularized_balanced,0.4958,0.0098,0.8426,0.0810
Communication,2,catboost_ordered_balanced,0.4740,0.0515,0.8364,0.0841
Communication,3,logistic_l2_balanced,0.4721,0.0329,0.8654,0.0832
Communication,4,ebm_main_effects_balanced,0.4307,0.0504,0.8381,0.0869
Communication,5,random_forest_balanced_leaf5,0.4226,0.0243,0.8235,0.0883
Communication,6,extra_trees_balanced_leaf5,0.4081,0.0330,0.8206,0.0890
Communication,7,svm_rbf_balanced,0.3271,0.0249,0.8073,0.0939
Cognitive,1,logistic_l2_balanced,0.5386,0.0166,0.8830,0.0672
Cognitive,2,xgboost_regularized_balanced,0.5223,0.0310,0.8836,0.0683
Cognitive,3,ebm_main_effects_balanced,0.5087,0.0194,0.8676,0.0706


## Complete performance comparison

In [4]:
metric_map = {'average_precision_pr_auc': 'PR-AUC', 'roc_auc': 'AUROC', 'accuracy': 'Accuracy', 'balanced_accuracy': 'Balanced accuracy', 'sensitivity': 'Sensitivity', 'specificity': 'Specificity', 'precision': 'PPV', 'negative_predictive_value': 'NPV', 'f1': 'F1', 'brier_score': 'Brier score'}
metric_rows = []
for key, domain in report['domains'].items():
    for name, candidate in domain['candidates'].items():
        summary = candidate['repeat_metric_summary']; row = {'Domain': domain_labels[key], 'Candidate': name}
        row.update({label: f"{summary[m]['mean']:.4f} ± {summary[m]['standard_deviation']:.4f}" for m, label in metric_map.items()}); metric_rows.append(row)
pd.DataFrame(metric_rows).style.hide(axis='index')

Domain,Candidate,PR-AUC,AUROC,Accuracy,Balanced accuracy,Sensitivity,Specificity,PPV,NPV,F1,Brier score
Communication,logistic_l2_balanced,0.4721 ± 0.0329,0.8654 ± 0.0114,0.7004 ± 0.0184,0.7884 ± 0.0187,0.9049 ± 0.0355,0.6720 ± 0.0214,0.2775 ± 0.0138,0.9808 ± 0.0069,0.4246 ± 0.0181,0.0832 ± 0.0022
Communication,svm_rbf_balanced,0.3271 ± 0.0249,0.8073 ± 0.0175,0.7572 ± 0.0117,0.7361 ± 0.0196,0.7082 ± 0.0469,0.7640 ± 0.0163,0.2944 ± 0.0124,0.9497 ± 0.0072,0.4157 ± 0.0171,0.0939 ± 0.0028
Communication,random_forest_balanced_leaf5,0.4226 ± 0.0243,0.8235 ± 0.0132,0.7060 ± 0.0246,0.7295 ± 0.0065,0.7607 ± 0.0377,0.6984 ± 0.0329,0.2604 ± 0.0118,0.9547 ± 0.0049,0.3874 ± 0.0096,0.0883 ± 0.0021
Communication,extra_trees_balanced_leaf5,0.4081 ± 0.0330,0.8206 ± 0.0117,0.7028 ± 0.0200,0.7348 ± 0.0205,0.7770 ± 0.0377,0.6925 ± 0.0230,0.2604 ± 0.0156,0.9572 ± 0.0068,0.3898 ± 0.0201,0.0890 ± 0.0021
Communication,xgboost_regularized_balanced,0.4958 ± 0.0098,0.8426 ± 0.0121,0.7364 ± 0.0413,0.7243 ± 0.0121,0.7082 ± 0.0440,0.7403 ± 0.0524,0.2788 ± 0.0318,0.9483 ± 0.0048,0.3983 ± 0.0272,0.0810 ± 0.0015
Communication,catboost_ordered_balanced,0.4740 ± 0.0515,0.8364 ± 0.0204,0.6636 ± 0.0273,0.7435 ± 0.0073,0.8492 ± 0.0355,0.6378 ± 0.0355,0.2465 ± 0.0117,0.9684 ± 0.0060,0.3817 ± 0.0116,0.0841 ± 0.0038
Communication,ebm_main_effects_balanced,0.4307 ± 0.0504,0.8381 ± 0.0199,0.7076 ± 0.0352,0.7587 ± 0.0247,0.8262 ± 0.0539,0.6911 ± 0.0433,0.2726 ± 0.0221,0.9665 ± 0.0090,0.4092 ± 0.0262,0.0869 ± 0.0040
Cognitive,logistic_l2_balanced,0.5386 ± 0.0166,0.8830 ± 0.0021,0.7596 ± 0.0180,0.8123 ± 0.0148,0.8792 ± 0.0316,0.7454 ± 0.0214,0.2912 ± 0.0149,0.9812 ± 0.0046,0.4372 ± 0.0174,0.0672 ± 0.0018
Cognitive,svm_rbf_balanced,0.3672 ± 0.0285,0.8545 ± 0.0085,0.8132 ± 0.0054,0.8090 ± 0.0156,0.8038 ± 0.0343,0.8143 ± 0.0071,0.3392 ± 0.0089,0.9723 ± 0.0046,0.4770 ± 0.0132,0.0812 ± 0.0040
Cognitive,random_forest_balanced_leaf5,0.3452 ± 0.0151,0.8601 ± 0.0083,0.7740 ± 0.0213,0.8171 ± 0.0175,0.8717 ± 0.0246,0.7624 ± 0.0235,0.3043 ± 0.0220,0.9804 ± 0.0038,0.4508 ± 0.0252,0.0787 ± 0.0018


## Configuration, calibration and threshold stability

In [5]:
stability_rows = []
for key, domain in report['domains'].items():
    for name in ('catboost_ordered_balanced', 'ebm_main_effects_balanced'):
        candidate = domain['candidates'][name]; stability_rows.append({'Domain': domain_labels[key], 'Candidate': name, 'Configuration selections': str(candidate['selected_configuration_counts']), 'Threshold mean': candidate['threshold_stability']['mean'], 'Threshold SD': candidate['threshold_stability']['standard_deviation'], 'Calibration coefficient mean': candidate['calibration_coefficient_stability']['mean'], 'Calibration coefficient SD': candidate['calibration_coefficient_stability']['standard_deviation'], 'Maximum explanation reconstruction error': candidate['maximum_calibrated_logit_reconstruction_error']})
pd.DataFrame(stability_rows).style.format({'Threshold mean': '{:.4f}', 'Threshold SD': '{:.4f}', 'Calibration coefficient mean': '{:.4f}', 'Calibration coefficient SD': '{:.4f}', 'Maximum explanation reconstruction error': '{:.3e}'}).hide(axis='index')

Domain,Candidate,Configuration selections,Threshold mean,Threshold SD,Calibration coefficient mean,Calibration coefficient SD,Maximum explanation reconstruction error
Communication,catboost_ordered_balanced,"{'catboost-ordered-balanced-depth3-v1': 19, 'catboost-ordered-balanced-depth4-v1': 6}",0.1151,0.0387,0.8950,0.1283,7.105e-15
Communication,ebm_main_effects_balanced,"{'ebm-main-effects-balanced-leaf16-v1': 11, 'ebm-main-effects-balanced-leaf8-v1': 14}",0.1165,0.0278,1.0158,0.2350,3.553e-15
Cognitive,catboost_ordered_balanced,"{'catboost-ordered-balanced-depth3-v1': 18, 'catboost-ordered-balanced-depth4-v1': 7}",0.1113,0.0318,0.8132,0.1394,4.441e-15
Cognitive,ebm_main_effects_balanced,"{'ebm-main-effects-balanced-leaf16-v1': 13, 'ebm-main-effects-balanced-leaf8-v1': 12}",0.1246,0.0302,1.0951,0.2246,2.665e-15
Social-emotional,catboost_ordered_balanced,"{'catboost-ordered-balanced-depth3-v1': 16, 'catboost-ordered-balanced-depth4-v1': 9}",0.1480,0.0573,0.8300,0.1044,3.553e-15
Social-emotional,ebm_main_effects_balanced,"{'ebm-main-effects-balanced-leaf16-v1': 10, 'ebm-main-effects-balanced-leaf8-v1': 15}",0.1318,0.0412,1.1360,0.1559,1.776e-15


## Paired uncertainty versus the historical selected candidate

In [6]:
uncertainty_rows = []
for key, domain in report['domains'].items():
    for candidate, uncertainty in domain['paired_uncertainty_vs_historical_reference'].items():
        for metric, values in uncertainty['differences'].items(): uncertainty_rows.append({'Domain': domain_labels[key], 'Reference': domain['historical_selected_reference'], 'Candidate': candidate, 'Metric': metric, 'Mean difference': values['mean'], '95% interval': f"[{values['confidence_interval_95'][0]:.4f}, {values['confidence_interval_95'][1]:.4f}]", 'Includes zero': values['confidence_interval_95'][0] <= 0 <= values['confidence_interval_95'][1]})
pd.DataFrame(uncertainty_rows).style.format({'Mean difference': '{:+.4f}'}).hide(axis='index')

Domain,Reference,Candidate,Metric,Mean difference,95% interval,Includes zero
Communication,xgboost_regularized_balanced,catboost_ordered_balanced,average_precision_pr_auc,-0.0074,"[-0.0403, 0.0242]",True
Communication,xgboost_regularized_balanced,catboost_ordered_balanced,roc_auc,+0.0068,"[-0.0043, 0.0196]",True
Communication,xgboost_regularized_balanced,catboost_ordered_balanced,brier_score,+0.0022,"[-0.0009, 0.0051]",True
Communication,xgboost_regularized_balanced,ebm_main_effects_balanced,average_precision_pr_auc,-0.0630,"[-0.1377, 0.0057]",True
Communication,xgboost_regularized_balanced,ebm_main_effects_balanced,roc_auc,+0.0027,"[-0.0266, 0.0309]",True
Communication,xgboost_regularized_balanced,ebm_main_effects_balanced,brier_score,+0.0045,"[-0.0005, 0.0098]",True
Cognitive,logistic_l2_balanced,catboost_ordered_balanced,average_precision_pr_auc,-0.0641,"[-0.1449, 0.0136]",True
Cognitive,logistic_l2_balanced,catboost_ordered_balanced,roc_auc,-0.0140,"[-0.0383, 0.0119]",True
Cognitive,logistic_l2_balanced,catboost_ordered_balanced,brier_score,+0.0064,"[0.0010, 0.0117]",False
Cognitive,logistic_l2_balanced,ebm_main_effects_balanced,average_precision_pr_auc,-0.0077,"[-0.0624, 0.0512]",True


## Scientific interpretation

In [7]:
leaders = {domain_labels[key]: domain['ranking'][0]['candidate'] for key, domain in report['domains'].items()}
display(Markdown('**Development-only conclusion.** Domain leaders were ' + ', '.join(f"{domain}: `{candidate}`" for domain, candidate in leaders.items()) + '. Rankings are adaptive development evidence, not independent confirmation. Small differences, calibration trade-offs, threshold instability and paired uncertainty must be considered before any model promotion. No new model was frozen or deployed.'))

**Development-only conclusion.** Domain leaders were Communication: `xgboost_regularized_balanced`, Cognitive: `logistic_l2_balanced`, Social-emotional: `xgboost_regularized_balanced`. Rankings are adaptive development evidence, not independent confirmation. Small differences, calibration trade-offs, threshold instability and paired uncertainty must be considered before any model promotion. No new model was frozen or deployed.

## Limitations

The same development data have informed earlier research decisions; positive outcome counts are small; calibration and threshold selection may be unstable; paired bootstrap intervals condition on averaged OOF predictions; multiple candidates and domains were inspected; and the previously evaluated test cohort was not accessed.